# RAG Evaluation

RAG evaluation is the process of measuring how well a Retrieval-Augmented
Generation system retrieves information and generates answers.

A RAG system can fail in different ways:

1. The retriever may retrieve the wrong documents.
2. The retrieved context may not be relevant to the question.
3. The LLM may fail to use the retrieved context correctly.
4. The generated answer may contain unsupported information.
5. The answer may not actually answer the user's question.

Therefore, evaluating only the final answer is not enough.

This notebook explores:

- Retrieval Precision
- Retrieval Recall
- Context Relevance
- Faithfulness
- Answer Relevance
- Basic RAG evaluation workflow


In [43]:
# Evaluation ko lagi sample dataset create garne
evaluation_data = [
    {
        "question": "What is RAG?",
        "relevant_documents": [
            "doc1",
            "doc2",
        ],  ##doc 1 doc 2 ko answer chai RAG ko barema ho
        "retrieved_documents": ["doc1", "doc2", "doc4"],
        "expected_answer": (
            "RAG combines information retrieval with language generation."
        ),
        "generated_answer": (
            "RAG combines information retrieval with language generation."
        ),
    },
    {
        "question": "What is a vector database?",
        "relevant_documents": ["doc3"],
        "retrieved_documents": ["doc3", "doc5"],
        "expected_answer": ("A vector database stores vector representations of data."),
        "generated_answer": (
            "A vector database stores vector representations of data."
        ),
    },
]

print(f"Evaluation samples: {len(evaluation_data)}")

Evaluation samples: 2


In [44]:
def retrieval_precision(
    relevant_documents,
    retrieved_documents,
):
    # Retrieved documents ma kati relevant chan calculate garne
    relevant_retrieved = set(relevant_documents).intersection(set(retrieved_documents))

    if not retrieved_documents:
        return 0.0

    return len(relevant_retrieved) / len(retrieved_documents)


def retrieval_recall(
    relevant_documents,
    retrieved_documents,
):
    # Relevant documents madhye kati retrieve bhayo calculate garne
    relevant_retrieved = set(relevant_documents).intersection(set(retrieved_documents))

    if not relevant_documents:
        return 0.0

    return len(relevant_retrieved) / len(relevant_documents)

In [45]:
# First evaluation sample ko retrieval metrics calculate garne
sample = evaluation_data[0]

precision = retrieval_precision(
    sample["relevant_documents"],
    sample["retrieved_documents"],
)

recall = retrieval_recall(
    sample["relevant_documents"],
    sample["retrieved_documents"],
)

print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.67
Recall: 1.00


In [46]:
# Local Ollama LLM use garna
from langchain_ollama import ChatOllama

# Evaluation ko lagi local LLM load garne
evaluator_llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)

print("Evaluation LLM loaded successfully!")

Evaluation LLM loaded successfully!


In [47]:
def evaluate_context_relevance(
    llm,
    question,
    context,
):
    # Context question ko lagi relevant cha ki chaina evaluate garne
    prompt = f"""
You are evaluating the relevance of retrieved context for a RAG system.

Question:
{question}

Retrieved Context:
{context}

Determine whether the context contains information useful for answering
the question.

Return only one of these labels:

RELEVANT
NOT_RELEVANT

Evaluation:
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [48]:
# Context relevance test garne
question = "What is RAG?"

context = """
Retrieval-Augmented Generation combines information retrieval
with language generation.
"""

result = evaluate_context_relevance(
    evaluator_llm,
    question,
    context,
)

print("Context relevance:", result)

Context relevance: RELEVANT


In [49]:
def evaluate_faithfulness(
    llm,
    context,
    answer,
):
    # Generated answer context bata supported cha ki chaina evaluate garne
    prompt = f"""
You are evaluating the faithfulness of an answer generated by a RAG system.

Context:
{context}

Answer:
{answer}

Determine whether the answer is fully supported by the provided context.

Return only one of these labels:

FAITHFUL
NOT_FAITHFUL

Evaluation:
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [50]:
# Faithfulness test garne
context = """
RAG combines information retrieval with language generation.
"""

answer = """
RAG combines information retrieval with language generation.
"""

result = evaluate_faithfulness(
    evaluator_llm,
    context,
    answer,
)

print("Faithfulness:", result)

Faithfulness: FAITHFUL


In [51]:
def evaluate_answer_relevance(
    llm,
    question,
    answer,
):
    # Generated answer question lai directly address garcha ki gardaina
    # evaluate garne
    prompt = f"""
You are evaluating the relevance of an answer generated by a RAG system.

Question:
{question}

Answer:
{answer}

Determine whether the answer directly addresses the question.

Return only one of these labels:

RELEVANT
NOT_RELEVANT

Evaluation:
"""

    response = llm.invoke(prompt)

    return response.content.strip()

In [52]:
# Answer relevance test garne
question = "What is a vector database?"

answer = """
A vector database stores vector representations of data.
"""

result = evaluate_answer_relevance(
    evaluator_llm,
    question,
    answer,
)

print("Answer relevance:", result)

Answer relevance: RELEVANT


In [53]:
def evaluate_rag_response(
    llm,
    question,
    relevant_documents,
    retrieved_documents,
    context,
    answer,
):
    # Retrieval precision calculate garne
    precision = retrieval_precision(
        relevant_documents,
        retrieved_documents,
    )

    # Retrieval recall calculate garne
    recall = retrieval_recall(
        relevant_documents,
        retrieved_documents,
    )

    # Context relevance evaluate garne
    context_relevance = evaluate_context_relevance(
        llm,
        question,
        context,
    )

    # Faithfulness evaluate garne
    faithfulness = evaluate_faithfulness(
        llm,
        context,
        answer,
    )

    # Answer relevance evaluate garne
    answer_relevance = evaluate_answer_relevance(
        llm,
        question,
        answer,
    )

    return {
        "retrieval_precision": precision,
        "retrieval_recall": recall,
        "context_relevance": context_relevance,
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
    }

In [54]:
# Complete RAG response evaluate garne
sample = evaluation_data[0]

context = """
Retrieval-Augmented Generation combines information retrieval
with language generation.
"""

results = evaluate_rag_response(
    llm=evaluator_llm,
    question=sample["question"],
    relevant_documents=sample["relevant_documents"],
    retrieved_documents=sample["retrieved_documents"],
    context=context,
    answer=sample["generated_answer"],
)

for metric, value in results.items():
    print(f"{metric}: {value}")

retrieval_precision: 0.6666666666666666
retrieval_recall: 1.0
context_relevance: RELEVANT
faithfulness: FAITHFUL
answer_relevance: RELEVANT
